In [1]:
# 计算 test_grid.ipynb 的“训练监督(render_3d) vs 输出图”MAE
import os
import re
import glob
from pathlib import Path

import cv2
import numpy as np

GT_DIR = "/data/test_grid4_from_test_grid4/render_3d"
OUT_DIR = "/home/viplab/liukejia/project/mv_model/results_grid4"
TARGET_ANGLES = [0, 90, 180, 270]

def parse_name_tokens(stem: str):
    parts = stem.split("_")
    if len(parts) < 4:
        return None
    try:
        nums = [int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])]
    except ValueError:
        return None
    num123 = f"{nums[0]}_{nums[1]}_{nums[2]}"
    return num123, nums[3]

def group_gt_files(gt_dir: str):
    files = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        files.extend(glob.glob(os.path.join(gt_dir, ext)))
    files = sorted(files)

    groups = {}
    for p in files:
        parsed = parse_name_tokens(Path(p).stem)
        if parsed is None:
            continue
        num123, num4 = parsed
        groups.setdefault(num123, []).append((num4, p))

    final = {}
    for num123, items in groups.items():
        angle_to_path = {}
        for num4, p in sorted(items, key=lambda x: (x[0], x[1])):
            if num4 in TARGET_ANGLES and num4 not in angle_to_path:
                angle_to_path[num4] = p
        if all(a in angle_to_path for a in TARGET_ANGLES):
            final[num123] = [angle_to_path[a] for a in TARGET_ANGLES]
    return final

def stitch_2x2_rgb(paths, image_size=512):
    sub = image_size // 2
    imgs = []
    for p in paths:
        bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        if bgr is None:
            return None
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (sub, sub), interpolation=cv2.INTER_AREA)
        imgs.append(rgb)

    top = np.concatenate([imgs[0], imgs[1]], axis=1)
    bottom = np.concatenate([imgs[2], imgs[3]], axis=1)
    return np.concatenate([top, bottom], axis=0)

def parse_num123_from_output_name(path: str):
    name = Path(path).stem
    m = re.match(r"num123_(-?\d+_-?\d+_-?\d+)_\d+", name)
    return None if m is None else m.group(1)

def mae(img1, img2):
    return float(np.mean(np.abs(img1.astype(np.float32) - img2.astype(np.float32))))

if not os.path.isdir(GT_DIR):
    print(f"未找到 GT_DIR: {GT_DIR}")
elif not os.path.isdir(OUT_DIR):
    print(f"未找到 OUT_DIR: {OUT_DIR}")
else:
    grouped_gt = group_gt_files(GT_DIR)
    outs = sorted(glob.glob(os.path.join(OUT_DIR, "*.png")))

    values = []
    skipped = 0

    for op in outs:
        num123 = parse_num123_from_output_name(op)
        if num123 is None or num123 not in grouped_gt:
            skipped += 1
            continue

        gt_rgb = stitch_2x2_rgb(grouped_gt[num123], image_size=512)
        if gt_rgb is None:
            skipped += 1
            continue

        out_bgr = cv2.imread(op, cv2.IMREAD_COLOR)
        if out_bgr is None:
            skipped += 1
            continue
        out_rgb = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)
        out_rgb = cv2.resize(out_rgb, (gt_rgb.shape[1], gt_rgb.shape[0]), interpolation=cv2.INTER_AREA)

        v = mae(gt_rgb, out_rgb)
        values.append((op, v))

    if len(values) == 0:
        print("没有可计算的配对样本，请检查路径和命名规则。")
    else:
        mae_list = [x[1] for x in values]
        print(f"有效样本数: {len(mae_list)}，跳过: {skipped}")
        print(f"平均 MAE: {np.mean(mae_list):.4f} (0-255, 越低越好)")
        print(f"中位数 MAE: {np.median(mae_list):.4f} (0-255, 越低越好)")
        print("前 5 个样本:")
        for p, v in values[:5]:
            print(f"  {Path(p).name}: {v:.4f}")

有效样本数: 9，跳过: 0
平均 MAE: 38.6983 (0-255, 越低越好)
中位数 MAE: 37.5234 (0-255, 越低越好)
前 5 个样本:
  num123_0_0_0_0001.png: 42.7538
  num123_1_0_0_0002.png: 36.0130
  num123_2_0_0_0003.png: 33.9871
  num123_3_0_0_0004.png: 37.5234
  num123_4_0_0_0005.png: 38.9818
